# Head 3 newline values and attention causality

This notebook tests whether Head 3's newline source tokens carry interchangeable value information, and whether attention to newline tokens predicts or causes newline outputs. It uses value interventions, attention-pattern redistribution, and attention removal on a fresh evaluation batch.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import roc_auc_score

PROJECT_ROOT = next(c for c in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd().parent.parent.parent] if (c / 'results/line_breaks_results').exists())
sys.path.insert(0, str(PROJECT_ROOT / 'src')) if str(PROJECT_ROOT / 'src') not in sys.path else None
import project_main.checkpoints as project_checkpoints
import project_main.data as project_data
import project_main.model as project_model
import project_main.tokens as project_tokens

RUN_DIR = PROJECT_ROOT / 'results/line_breaks_results'
with open(RUN_DIR / 'config.json') as f: cfg = json.load(f)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
vocab = project_tokens.build_vocab(cfg['task'])
model = project_model.build_model(cfg=cfg, device=device)
checkpoint = project_checkpoints.load_checkpoint(path=RUN_DIR / 'checkpoints/final.pt', model=model, map_location=device)
model.eval()
HEAD = 3
FINAL_LAYER = cfg['model']['n_layers'] - 1
PATTERN_SITE = f'blocks.{FINAL_LAYER}.attn.hook_pattern'
VALUE_SITE = f'blocks.{FINAL_LAYER}.attn.hook_v'
BATCH_SIZE = 2048
print(f'Loaded checkpoint step {checkpoint["step"]} on {device}; final layer={FINAL_LAYER}, head={HEAD}')

Building vocab...
('BOS', '_NEWLINE_')
Loaded checkpoint step 10000 on cpu; final layer=2, head=3


In [2]:
batch = project_data.make_batch(batch_size=BATCH_SIZE, vocab=vocab, task_cfg=cfg['task'], device=device, seed=29001)
tokens, targets = batch.tokens, batch.targets
with torch.no_grad():
    baseline_logits, cache = model.run_with_cache(tokens, prepend_bos=False, names_filter=lambda n: n in {PATTERN_SITE, VALUE_SITE})
pattern = cache[PATTERN_SITE].detach()
values = cache[VALUE_SITE].detach()
W_O_HEAD = model.W_O[FINAL_LAYER, HEAD].detach()
newline_id = vocab.newline_id
source_newline = tokens == newline_id
target_newline = targets == newline_id
torch.manual_seed(29003)
shuffle_permutations = []
for b in range(tokens.shape[0]):
    pos = torch.where(source_newline[b])[0]
    shuffle_permutations.append(torch.randperm(len(pos), device=pos.device) if len(pos) else None)
print(f'Batch shape={tuple(tokens.shape)}; source newline rate={source_newline.float().mean().item():.3f}; target newline rate={target_newline.float().mean().item():.3f}')

Batch shape=(2048, 150); source newline rate=0.053; target newline rate=0.054


In [3]:
def metrics(logits):
    pred = logits.argmax(-1)
    pred_nl = pred == newline_id
    other = logits.masked_fill(torch.nn.functional.one_hot(torch.full_like(pred, newline_id), logits.shape[-1]).bool(), -torch.inf).max(-1).values
    margin = logits[..., newline_id] - other
    tp = (pred_nl & target_newline).sum().item()
    return {'accuracy': (pred == targets).float().mean().item(), 'newline_precision': tp / pred_nl.sum().item(), 'newline_recall': tp / target_newline.sum().item(), 'newline_frequency': pred_nl.float().mean().item(), 'newline_margin': margin, 'predictions': pred}

base = metrics(baseline_logits)
def run(site, fn):
    with torch.no_grad(): return model.run_with_hooks(tokens, prepend_bos=False, fwd_hooks=[(site, fn)])

def value_patch(kind):
    def patch(v, hook):
        v = v.clone()
        for b in range(v.shape[0]):
            pos = torch.where(source_newline[b])[0]
            if len(pos) == 0: continue
            if kind == 'mean': v[b, pos, HEAD] = v[b, pos, HEAD].mean(0, keepdim=True)
            elif kind == 'shuffle':
                v[b, pos, HEAD] = v[b, pos[shuffle_permutations[b]], HEAD]
            elif kind == 'zero': v[b, pos, HEAD] = 0
        return v
    return patch

def pattern_patch(kind):
    def patch(p, hook):
        p = p.clone()
        for b in range(p.shape[0]):
            for q in range(p.shape[2]):
                pos = torch.where(source_newline[b, :q])[0]
                if len(pos) == 0: continue
                total = p[b, HEAD, q, pos].sum(); p[b, HEAD, q, pos] = 0
                if kind == 'uniform': p[b, HEAD, q, pos] = total / len(pos)
                elif kind == 'previous': p[b, HEAD, q, pos[-1]] = total
                elif kind == 'remove': p[b, HEAD, q] /= p[b, HEAD, q].sum()
        return p
    return patch

def summarize(condition, logits):
    m = metrics(logits)
    return {'condition': condition, **{k: v for k, v in m.items() if not isinstance(v, torch.Tensor)}, 'mean_abs_logit_change': (logits-baseline_logits).abs().mean().item(), 'prediction_flip_rate': (m['predictions'] != base['predictions']).float().mean().item(), 'mean_abs_margin_change': (m['newline_margin']-base['newline_margin']).abs().mean().item()}
print({k: v for k, v in base.items() if not isinstance(v, torch.Tensor)})

{'accuracy': 0.0646451786160469, 'newline_precision': 0.52198949566538, 'newline_recall': 0.999454776761374, 'newline_frequency': 0.10288411378860474}


## Do newline source values carry the same information?

Attention is held fixed. Within each sequence, newline values are replaced by their mean, permuted among newline positions, or zeroed. Small mean/shuffle effects indicate interchangeability; zeroing measures the total contribution of newline values.

In [4]:
value_table = pd.DataFrame([summarize('baseline', baseline_logits)] + [summarize(f'value_{k}', run(VALUE_SITE, value_patch(k))) for k in ('mean', 'shuffle', 'zero')])
display(value_table)

rows = []
for b in range(tokens.shape[0]):
    for q in range(tokens.shape[1]):
        pos = torch.where(source_newline[b, :q])[0]
        w = pattern[b, HEAD, q, pos] if len(pos) else torch.zeros(1, device=device)
        rows.append({'newline_attention': w.sum().item(), 'previous_newline_attention': w[-1].item() if len(pos) else 0, 'target_newline': bool(target_newline[b,q]), 'predicted_newline': bool(base['predictions'][b,q] == newline_id), 'margin': base['newline_margin'][b,q].item()})
attention_df = pd.DataFrame(rows)
for x in ('newline_attention', 'previous_newline_attention'):
    for y in ('target_newline', 'predicted_newline'):
        print(x, y, 'AUC=', roc_auc_score(attention_df[y], attention_df[x]))
display(attention_df[['newline_attention','previous_newline_attention','margin']].corr())
attention_df['bin'] = pd.qcut(attention_df.newline_attention, 10, duplicates='drop')
display(attention_df.groupby('bin', observed=True).agg(n=('margin','size'), attention=('newline_attention','mean'), target_newline=('target_newline','mean'), predicted_newline=('predicted_newline','mean'), margin=('margin','mean')))

,condition,accuracy,newline_precision,newline_recall,newline_frequency,mean_abs_logit_change,prediction_flip_rate,mean_abs_margin_change
0,baseline,0.064645,0.521989,0.999455,0.102884,0.000000,0.000000,0.000000
1,value_mean,0.062679,0.475928,0.964197,0.108861,0.020080,0.040921,0.320913
2,value_shuffle,0.062031,0.470898,0.952808,0.108724,0.023279,0.043112,0.369305
3,value_zero,0.060247,0.494193,0.917671,0.099779,0.016804,0.051491,0.269074


newline_attention target_newline AUC= 0.8634309753664612
newline_attention predicted_newline AUC= 0.5875763473916709
previous_newline_attention target_newline AUC= 0.875689386475775
previous_newline_attention predicted_newline AUC= 0.5987783130942903


,newline_attention,previous_newline_attention,margin
newline_attention,1.000000,0.818706,0.460163
previous_newline_attention,0.818706,1.000000,0.713504
margin,0.460163,0.713504,1.000000


,n,attention,target_newline,predicted_newline,margin
bin,,,,,
"(-0.001, 0.0052]",61440,0.001188,0.033415,0.214681,-5.330264
"(0.0052, 0.00993]",30720,0.007486,0.000098,0.000098,-8.232435
"(0.00993, 0.0163]",30720,0.012950,0.000033,0.000033,-8.240490
"(0.0163, 0.0263]",30720,0.020942,0.000098,0.000098,-8.243671
"(0.0263, 0.0442]",30720,0.034317,0.000098,0.000098,-8.250481
"(0.0442, 0.0843]",30720,0.060568,0.000065,0.000130,-8.247374
"(0.0843, 0.371]",30720,0.181151,0.000618,0.002507,-8.229450
"(0.371, 0.907]",30720,0.647402,0.041016,0.085384,-7.167611
"(0.907, 1.0]",30720,0.977479,0.428483,0.511133,1.222566


## Does concentration among newline keys matter?

These interventions preserve the total Head 3 attention mass on newline keys, but put it all on the previous newline or spread it uniformly across previous newlines. Removal renormalizes the remaining attention. A difference between previous and uniform means that which newline is attended to matters, not just total newline mass.

In [5]:
pattern_table = pd.DataFrame([summarize('baseline', baseline_logits)] + [summarize(f'attention_{k}', run(PATTERN_SITE, pattern_patch(k))) for k in ('previous', 'uniform', 'remove')])
display(pattern_table)

rows = []
for b in range(tokens.shape[0]):
    for q in range(tokens.shape[1]):
        pos = torch.where(source_newline[b, :q])[0]
        if len(pos) == 0: continue
        w = pattern[b, HEAD, q, pos]
        actual = (w[:,None] * values[b, pos, HEAD]).sum(0) @ W_O_HEAD
        concentrated = (w.sum() * values[b, pos[-1], HEAD]) @ W_O_HEAD
        rows.append({'newline_attention': w.sum().item(), 'difference_norm': (actual-concentrated).norm().item(), 'relative_difference': (actual-concentrated).norm().item()/(actual.norm().item()+1e-8), 'target_newline': bool(target_newline[b,q])})
display(pd.DataFrame(rows).groupby('target_newline').agg(n=('difference_norm','size'), newline_attention=('newline_attention','mean'), difference_norm=('difference_norm','mean'), relative_difference=('relative_difference','mean')))
print('Interpretation: compare value_mean/value_shuffle for interchangeability, attention_previous vs attention_uniform for concentration effects, and attention_remove for causal necessity.')

,condition,accuracy,newline_precision,newline_recall,newline_frequency,mean_abs_logit_change,prediction_flip_rate,mean_abs_margin_change
0,baseline,0.064645,0.521989,0.999455,0.102884,0.000000,0.000000,0.000000
1,attention_previous,0.064590,0.493568,0.999455,0.108809,0.007635,0.043040,0.122104
2,attention_uniform,0.060469,0.522856,0.920882,0.094639,0.011654,0.023910,0.192263
3,attention_remove,0.063200,0.516489,0.973466,0.101276,0.014919,0.046729,0.237501


,n,newline_attention,difference_norm,relative_difference
target_newline,,,,
False,253552,0.180467,0.666586,1.284928
True,14459,0.967038,0.610709,0.166056


Interpretation: compare value_mean/value_shuffle for interchangeability, attention_previous vs attention_uniform for concentration effects, and attention_remove for causal necessity.


## Newline-only conditional effects

The following table avoids conflating ordinary-token behavior with newline behavior. It reports effects among (a) positions that baseline predicts as newline and (b) positions whose true target is newline. `predicted_to_nonnewline` measures newline predictions lost under the intervention; `nonnewline_to_predicted` measures new newline predictions created. Margin changes are for the newline-vs-best-nonnewline logit margin.

In [6]:
def conditional_effects(condition, logits):
    m = metrics(logits)
    pred0 = base['predictions'] == newline_id
    pred1 = m['predictions'] == newline_id
    rows = []
    for subset, mask in [('baseline_predicted_newline', pred0), ('true_target_newline', target_newline)]:
        n = mask.sum().item()
        rows.append({
            'condition': condition, 'subset': subset, 'n': n,
            'prediction_changed': (pred1[mask] != pred0[mask]).float().mean().item(),
            'predicted_to_nonnewline': (pred0[mask] & ~pred1[mask]).float().mean().item(),
            'nonnewline_to_predicted': (~pred0[mask] & pred1[mask]).float().mean().item(),
            'newline_margin_change': (m['newline_margin'][mask] - base['newline_margin'][mask]).mean().item(),
            'newline_margin_abs_change': (m['newline_margin'][mask] - base['newline_margin'][mask]).abs().mean().item(),
        })
    return rows

conditional_rows = conditional_effects('baseline', baseline_logits)
for kind in ('mean', 'shuffle', 'zero'):
    conditional_rows += conditional_effects(f'value_{kind}', run(VALUE_SITE, value_patch(kind)))
for kind in ('previous', 'uniform', 'remove'):
    conditional_rows += conditional_effects(f'attention_{kind}', run(PATTERN_SITE, pattern_patch(kind)))
conditional_table = pd.DataFrame(conditional_rows)
display(conditional_table)
print('Interpretation: compare the two subsets within each intervention. If newline values or newline attention have the same impact in both senses, these conditional rows should be similar; differences indicate target-conditioned versus prediction-conditioned effects.')

,condition,subset,n,prediction_changed,predicted_to_nonnewline,nonnewline_to_predicted,newline_margin_change,newline_margin_abs_change
0,baseline,baseline_predicted_newline,31606,0.000000,0.000000,0.000000,0.000000,0.000000
1,baseline,true_target_newline,16507,0.000000,0.000000,0.000000,0.000000,0.000000
2,value_mean,baseline_predicted_newline,31606,0.032715,0.032715,0.000000,0.509933,1.874208
3,value_mean,true_target_newline,16507,0.035379,0.035318,0.000061,0.357219,2.589193
4,value_shuffle,baseline_predicted_newline,31606,0.041638,0.041638,0.000000,0.281633,2.111130
5,value_shuffle,true_target_newline,16507,0.046647,0.046647,0.000000,-0.088705,2.918834
6,value_zero,baseline_predicted_newline,31606,0.069860,0.069860,0.000000,-0.632427,1.888232
7,value_zero,true_target_newline,16507,0.081905,0.081844,0.000061,-1.178629,2.830054
8,attention_previous,baseline_predicted_newline,31606,0.000000,0.000000,0.000000,0.423595,0.467262
9,attention_previous,true_target_newline,16507,0.000000,0.000000,0.000000,0.562448,0.646044


Interpretation: compare the two subsets within each intervention. If newline values or newline attention have the same impact in both senses, these conditional rows should be similar; differences indicate target-conditioned versus prediction-conditioned effects.


## Logit differences at newline-relevant positions

This repeats the intervention comparison using only positions where either the baseline prediction or the true target is `_NEWLINE_`. The main quantity is the change in the newline-vs-best-non-newline logit margin; this reveals effects even when the argmax prediction does not flip.

In [7]:
relevant = (base['predictions'] == newline_id) | target_newline
def logit_diff_row(condition, logits):
    m = metrics(logits)
    delta_margin = (m['newline_margin'] - base['newline_margin'])[relevant]
    delta_newline_logit = (logits[..., newline_id] - baseline_logits[..., newline_id])[relevant]
    return {
        'condition': condition, 'n_relevant': int(relevant.sum().item()),
        'mean_delta_newline_logit': delta_newline_logit.mean().item(),
        'mean_abs_delta_newline_logit': delta_newline_logit.abs().mean().item(),
        'mean_delta_newline_margin': delta_margin.mean().item(),
        'mean_abs_delta_newline_margin': delta_margin.abs().mean().item(),
        'median_delta_newline_margin': delta_margin.median().item(),
        'p10_delta_newline_margin': torch.quantile(delta_margin, 0.10).item(),
        'p90_delta_newline_margin': torch.quantile(delta_margin, 0.90).item(),
    }

logit_diff_rows = [logit_diff_row('baseline', baseline_logits)]
for kind in ('mean', 'shuffle', 'zero'):
    logit_diff_rows.append(logit_diff_row(f'value_{kind}', run(VALUE_SITE, value_patch(kind))))
for kind in ('previous', 'uniform', 'remove'):
    logit_diff_rows.append(logit_diff_row(f'attention_{kind}', run(PATTERN_SITE, pattern_patch(kind))))
display(pd.DataFrame(logit_diff_rows))
print('Positive margin change favors newline; negative margin change disfavors newline. The relevant set is the union of baseline-newline and true-newline positions.')

,condition,n_relevant,mean_delta_newline_logit,mean_abs_delta_newline_logit,mean_delta_newline_margin,mean_abs_delta_newline_margin,median_delta_newline_margin,p10_delta_newline_margin,p90_delta_newline_margin
0,baseline,31615,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000
1,value_mean,31615,0.510646,1.830695,0.509868,1.873754,0.0,-2.301573,4.817830
2,value_shuffle,31615,0.292608,2.061948,0.281553,2.110529,0.0,-3.108600,5.123096
3,value_zero,31615,-0.606802,1.833672,-0.632183,1.887758,0.0,-5.193198,2.530189
4,attention_previous,31615,0.413257,0.454740,0.423474,0.467129,0.0,-0.037898,1.634247
5,attention_uniform,31615,-1.356976,1.443689,-1.398136,1.489650,0.0,-5.368039,0.000000
6,attention_remove,31615,-0.115809,1.471147,-0.131356,1.510616,0.0,-2.718762,3.082534


Positive margin change favors newline; negative margin change disfavors newline. The relevant set is the union of baseline-newline and true-newline positions.


## Logit differences restricted to true newline targets

This is the same calculation restricted to positions where the true target is `_NEWLINE_`. It measures how each intervention changes the newline logit and the newline-vs-best-non-newline margin specifically at genuine line-break decisions.

In [8]:
relevant = target_newline
target_logit_rows = [logit_diff_row('baseline', baseline_logits)]
for kind in ('mean', 'shuffle', 'zero'):
    target_logit_rows.append(logit_diff_row(f'value_{kind}', run(VALUE_SITE, value_patch(kind))))
for kind in ('previous', 'uniform', 'remove'):
    target_logit_rows.append(logit_diff_row(f'attention_{kind}', run(PATTERN_SITE, pattern_patch(kind))))
display(pd.DataFrame(target_logit_rows))
print('All rows are now computed only over true newline targets; positive margin change favors newline.')

,condition,n_relevant,mean_delta_newline_logit,mean_abs_delta_newline_logit,mean_delta_newline_margin,mean_abs_delta_newline_margin,median_delta_newline_margin,p10_delta_newline_margin,p90_delta_newline_margin
0,baseline,16507,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,value_mean,16507,0.368832,2.532788,0.357219,2.589193,0.000000,-3.716122,5.424303
2,value_shuffle,16507,-0.060612,2.853317,-0.088705,2.918834,0.000000,-4.931146,5.703603
3,value_zero,16507,-1.137130,2.753922,-1.178629,2.830054,0.000000,-7.249487,3.205090
4,attention_previous,16507,0.552323,0.631761,0.562448,0.646044,0.128321,-0.153515,1.866730
5,attention_uniform,16507,-2.091170,2.255741,-2.147959,2.321673,-0.784492,-6.597600,0.283056
6,attention_remove,16507,0.103203,1.981441,0.087549,2.024848,0.000000,-2.942613,4.085834


All rows are now computed only over true newline targets; positive margin change favors newline.


## Overall newline metrics

This final table reports the model-wide newline precision and recall for the baseline and every intervention. The delta columns are percentage-point changes relative to the baseline.

In [9]:
metric_rows = []
conditions = [('baseline', baseline_logits)]
conditions += [(f'value_{kind}', run(VALUE_SITE, value_patch(kind))) for kind in ('mean', 'shuffle', 'zero')]
conditions += [(f'attention_{kind}', run(PATTERN_SITE, pattern_patch(kind))) for kind in ('previous', 'uniform', 'remove')]
for condition, logits in conditions:
    m = metrics(logits)
    metric_rows.append({'condition': condition, 'newline_precision': m['newline_precision'], 'newline_recall': m['newline_recall'], 'newline_frequency': (m['predictions'] == newline_id).float().mean().item()})
metric_table = pd.DataFrame(metric_rows)
baseline_precision = metric_table.loc[metric_table.condition == 'baseline', 'newline_precision'].iloc[0]
baseline_recall = metric_table.loc[metric_table.condition == 'baseline', 'newline_recall'].iloc[0]
metric_table['precision_delta_pp'] = 100 * (metric_table['newline_precision'] - baseline_precision)
metric_table['recall_delta_pp'] = 100 * (metric_table['newline_recall'] - baseline_recall)
display(metric_table)
print('Precision and recall are fractions in the first columns; delta columns are percentage points relative to baseline.')

,condition,newline_precision,newline_recall,newline_frequency,precision_delta_pp,recall_delta_pp
0,baseline,0.521989,0.999455,0.102884,0.000000,0.000000
1,value_mean,0.475928,0.964197,0.108861,-4.606102,-3.525777
2,value_shuffle,0.470898,0.952808,0.108724,-5.109129,-4.664688
3,value_zero,0.494193,0.917671,0.099779,-2.779662,-8.178349
4,attention_previous,0.493568,0.999455,0.108809,-2.842161,0.000000
5,attention_uniform,0.522856,0.920882,0.094639,0.086676,-7.857273
6,attention_remove,0.516489,0.973466,0.101276,-0.550068,-2.598897


Precision and recall are fractions in the first columns; delta columns are percentage points relative to baseline.
